In [1]:
import enum
from random import random
from re import S
from xml.etree.ElementInclude import include
from sklearn import metrics
import tensorflow as tf
from tensorflow.keras import callbacks, Input
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, AveragePooling2D, ReLU, Flatten, Dense, GlobalAveragePooling2D, BatchNormalization, Dropout, Lambda, ZeroPadding2D, concatenate, Average
from tensorflow.keras.optimizers import SGD, Adam, RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
from tensorflow.keras.losses import MeanSquaredError, MeanAbsoluteError, CategoricalCrossentropy
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.applications import VGG16

from sklearn.model_selection import KFold

from keras.utils.vis_utils import plot_model

from collections import Counter

import scipy.ndimage as sci

import numpy as np
from sklearn.metrics import *
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
import os
import cv2 as cv

import random

import ranges_of_age as roa

import sys

from datetime import datetime

import functions as fn

2024-02-07 12:53:53.777147: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2024-02-07 12:53:53.777163: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


# Instrucciones de USO

El notebook esa controlado por 3 variables principales:
- **COLORMODE**: ["rgb"/"grayscale"] controla si utiliza la representación RGB o la representación en escala de grises (SDM, NDM y GNDM por separado)
- **RESNET**: controla si utiliza la red RESNET-CNN (solo con rgb) o la red PANORAMA-CNN
- **MODEL**: carga el modelo entrenado. Los modelos tiene todos la nomenclatura DDHHMM. Se nombran en función del día DD, hora HH y minuto MM en el que comenzó el entrenamiento. Por defecto se carga el 221512. Los modelos entrenados estan en la carpeta execution.

In [2]:
COLORMODE = "rgb"
RESNET = False
MODEL = 221829
path_to_trained_models='modelos_entrenados'

In [3]:
VERBOSE = 2
mse = []
r2 = []
rmse = []
mae = []

IMG_SHAPE = (36,108)
depth = 3
if RESNET:
    cnn = fn.create_CNN_Resnet
    model_weights = os.path.join(path_to_trained_models, str(MODEL), 'model_rgb_resnet/model_rgb_resnet')
else:
    cnn = fn.create_CNN
    model_weights = os.path.join(path_to_trained_models, str(MODEL), 'model_rgb_panoramacnn/model_rgb_panoramacnn')

print(model_weights)

modelos_entrenados/221829/model_rgb_panoramacnn/model_rgb_panoramacnn


In [4]:
cnn_0 = cnn(0,IMG_SHAPE[0],IMG_SHAPE[1],depth)
cnn_1 = cnn(1,IMG_SHAPE[0],IMG_SHAPE[1],depth)
cnn_2 = cnn(2,IMG_SHAPE[0],IMG_SHAPE[1],depth)

2024-02-07 12:53:55.334394: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2024-02-07 12:53:55.334632: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/manza/python-envs/tfg_env/lib/python3.8/site-packages/cv2/../../lib64:
2024-02-07 12:53:55.334690: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/manza/python-envs/tfg_env/lib/python3.8/site-packages/cv2/../../lib64:
2024-02-07 12:53:55.334740: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasL

# Eliminar la ultima capa

La siguiente celda junta el output de las 3 redes en una capa Average de Keras.

In [5]:
# Normalmente utilizamos una capa final Average, pero para sacar cada predicción por separado, la quitamos.
# x = Average()([cnn_0.output, cnn_1.output, cnn_2.output])
x = [cnn_0.output, cnn_1.output, cnn_2.output]

In [6]:
model = Model(inputs=[cnn_0.input, cnn_1.input, cnn_2.input], outputs=x)

optimizer = Adam()

model.compile(loss=MeanAbsoluteError(), optimizer=optimizer, metrics=['mae'])
model.summary()

Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 108, 36, 3)  0           []                               
                                ]                                                                 
                                                                                                  
 input_2 (InputLayer)           [(None, 108, 36, 3)  0           []                               
                                ]                                                                 
                                                                                                  
 input_3 (InputLayer)           [(None, 108, 36, 3)  0           []                               
                                ]                                                           

In [7]:
tf_version = tf.__version__
if tf_version != "2.9.0":
    print("Error: Tensorflow version must be 2.9.0 and current version is {}".format(tf_version),file=sys.stderr)
    exit(1)

model.load_weights(model_weights)

In [8]:
folder = "../pubis_data_proc_25_panorama_norm/"
name = "177_Izq"

In [9]:
filename = name+"/"+name+"_0"
test = [[filename+'_panorama_ext_X.png',filename+'_panorama_ext_Y.png',filename+'_panorama_ext_Z.png',0]]
datagen = ImageDataGenerator()
datagen_test = fn.image_generator(test, folder, 1, datagen, IMG_SHAPE, colormode=COLORMODE, shuffle=False, weights=False)

# estimation = model.predict(datagen_test, verbose=VERBOSE, steps=len(test))
# est = estimation.flatten()
# print("Age estimation: {}".format(est[0]))

estimation = model.predict(datagen_test, verbose=VERBOSE, steps=len(test))
for i, est in enumerate(estimation):
    print("CNN {} - Age estimation: {}".format(i, est[0][0]))

1/1 - 0s - 193ms/epoch - 193ms/step
CNN 0 - Age estimation: 0.0
CNN 1 - Age estimation: 58.491512298583984
CNN 2 - Age estimation: 0.0
